# Movie Data Analysis

This Jupyter Notebook performs a comprehensive analysis of movie data fetched from the TMDB API. It uses 

functions defined in `tmdb_functions.py` to fetch, clean, and analyze data, calculate KPIs, perform advanced 

filtering, compare franchises vs. standalone movies, analyze franchises and directors, and generate 

visualizations.


In [ ]:
# Import necessary libraries
import os
import pandas as pd
import matplotlib.pyplot as plt




In [ ]:
# Import functions from tmdb_functions.py
from tmdb_functions import (
    get_api_key,                     # Function to retrieve the API key
    fetch_movie_data,               # Function to fetch movie data from TMDB API
    save_df,                        # Function to save dataframe to file
    load_df,                        # Function to load dataframe from file
    clean_df,                       # Function to clean the dataframe
    kpi_ranking,                    # Function to calculate key performance indicators
    advanced_search,                # Function to perform advanced movie filtering
    franchise_vs_standalone,        # Function to compare franchise and standalone movies
    analyze_franchise,              # Function to analyze franchise performance
    analyze_directors,              # Function to evaluate director performance
    plot_revenue_vs_budget,         # Function to plot revenue vs. budget
    plot_roi_by_genre,              # Function to visualize ROI by genre
    plot_popularity_vs_rating,      # Function to plot popularity vs. rating
    plot_yearly_box_office,         # Function to show trends over years
    plot_franchise_vs_standalone    # Function to compare franchises vs standalone movies graphically
)


## Fetching and Saving Raw Movie Data

- **Get API key:** Retrieves the TMDB API key from environment variables using `get_api_key()`.

- **Define movie IDs:** Creates a list of specific TMDB movie IDs to analyze.

- **Fetch data:** Uses the custom function `fetch_movie_data()` to collect metadata for each movie.

- **Save to CSV:** Saves the raw movie data into a file called `raw_movie_data_new.csv`.

- **Preview data:** Displays the first few records of the fetched raw data using `.head()`.


In [ ]:
# Get API
get_api_key()

# Define movie IDs and fetch data
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
             420818, 24428, 168259, 99861, 284054, 12445,
             181808, 330457, 351286, 109445, 321612, 260513]

# Fetch movie data
raw_data = fetch_movie_data(movie_ids)

# Save raw data to CSV
save_df(raw_data, 'raw_movie_data_new.csv')

# Display first few rows
raw_data.head()

## Cleaning the Movie Dataset

- **Clean raw data:** Applies `clean_df()` to standardize column names, fix formatting issues, and convert data types.
- **Replace 'No Data':** Converts placeholder strings like "No Data" in the 'overview' and 'tagline' columns to missing values (`pd.NA`).
- **Remove duplicates and missing values:** Drops duplicates and any rows that do not contain valid `id` or `title`.
- **Drop sparse rows:** Removes rows with fewer than 10 non-null entries.
- **Filter for released movies only:** Keeps only movies with a status of `'Released'`, then removes the status column.
- **Save cleaned data:** Exports the cleaned dataset as `cleaned_movie_data.csv`.
- **Preview cleaned data:** Displays the first few records of the cleaned dataset.


In [ ]:
# Clean the data
cleaned_data = clean_df(raw_data)

# Additional cleaning steps
cleaned_data['overview'] = cleaned_data['overview'].replace('No Data', pd.NA)
cleaned_data['tagline'] = cleaned_data['tagline'].replace('No Data', pd.NA)
cleaned_data = cleaned_data.drop_duplicates().dropna(subset=['id', 'title'])
cleaned_data = cleaned_data.dropna(thresh=10)
if 'status' in cleaned_data:
    cleaned_data = cleaned_data[cleaned_data['status'] == 'Released']
    cleaned_data = cleaned_data.drop(columns=['status'])

# Save cleaned data to CSV
save_df(cleaned_data, 'cleaned_movie_data.csv')

# Display first few rows
cleaned_data.head()

## Ranking Movies by Key Metrics

This section ranks movies based on various performance indicators using the `kpi_ranking()` function:

- **Top by Revenue:** Shows the 5 movies with the highest revenue.
- **Top by Budget:** Displays the top 5 movies with the highest production budgets.
- **Top by Profit:** Ranks movies by the difference between revenue and budget.
- **Bottom by Profit:** Shows the 5 least profitable movies.
- **Top ROI:** Displays top 5 movies with highest Return on Investment (ROI), filtering out movies with budget < 10M.
- **Bottom ROI:** Shows the worst-performing 5 movies by ROI (budget ≥ 10M).
- **Most Voted:** Lists movies with the highest number of audience votes.
- **Highest Rated:** Displays the best-rated movies (only those with at least 10 votes).
- **Lowest Rated:** Displays the lowest-rated movies (with ≥ 10 votes).
- **Most Popular:** Shows the most popular movies based on TMDB's popularity score.


In [ ]:
# Calculate KPIs
print("Top 5 Movies by Revenue:")
print(kpi_ranking(cleaned_data, 'revenue_millions', n=5)[['title', 'revenue_millions']])

print("\nTop 5 Movies by Budget:")
print(kpi_ranking(cleaned_data, 'budget_millions', n=5)[['title', 'budget_millions']])

print("\nTop 5 Movies by Profit:")
print(kpi_ranking(cleaned_data, 'profit', n=5)[['title', 'profit']])

print("\nBottom 5 Movies by Profit:")
print(kpi_ranking(cleaned_data, 'profit', n=5, top=False)[['title', 'profit']])

print("\nTop 5 Movies by ROI (Budget >= 10M):")
print(kpi_ranking(cleaned_data, 'roi', n=5, filter_col='budget_millions', filter_val=10)[['title', 'roi']])

print("\nBottom 5 Movies by ROI (Budget >= 10M):")
print(kpi_ranking(cleaned_data, 'roi', n=5, top=False, filter_col='budget_millions', filter_val=10)[['title', 'roi']])

print("\nMost Voted Movies:")
print(kpi_ranking(cleaned_data, 'vote_count', n=5)[['title', 'vote_count']])

print("\nHighest Rated Movies (>= 10 votes):")
print(kpi_ranking(cleaned_data, 'vote_average', n=5, filter_col='vote_count', filter_val=10)[['title', 'vote_average']])

print("\nLowest Rated Movies (>= 10 votes):")
print(kpi_ranking(cleaned_data, 'vote_average', n=5, top=False, filter_col='vote_count', filter_val=10)[['title', 'vote_average']])

print("\nMost Popular Movies:")
print(kpi_ranking(cleaned_data, 'popularity', n=5)[['title', 'popularity']])

## Advanced Search for Specific Movies

This section filters movies based on custom conditions using the `advanced_search()` function:

- **Search 1:** Retrieves Science Fiction + Action movies starring Bruce Willis, sorted by rating.

- **Search 2:** Finds movies featuring Uma Thurman and directed by Quentin Tarantino, sorted by runtime.


In [ ]:
# Advanced filtering
# Search 1: Best-rated Science Fiction Action movies starring Bruce Willis
sci_fi_action_willis = advanced_search(
    cleaned_data,
    genre_keywords='Science Fiction|Action',
    cast_keywords='Bruce Willis',
    sort_by='vote_average',
    ascending=False
)
print("Science Fiction Action Movies Starring Bruce Willis (Sorted by Rating):")
print(sci_fi_action_willis[['title', 'genres', 'cast', 'vote_average']])

# Search 2: Movies starring Uma Thurman, directed by Quentin Tarantino
thurman_tarantino = advanced_search(
    cleaned_data,
    cast_keywords='Uma Thurman',
    director_keywords='Quentin Tarantino',
    sort_by='runtime',
    ascending=True
)
print("\nMovies Starring Uma Thurman Directed by Quentin Tarantino (Sorted by Runtime):")
print(thurman_tarantino[['title', 'cast', 'director', 'runtime']])

## Franchise vs. Standalone Movie Analysis

- **Compare performance:** Uses `franchise_vs_standalone()` to compare the average performance of movie franchises versus standalone films.

- **Output includes:** metrics like average revenue, budget, profit, and rating.


In [ ]:
# Franchise vs. standalone comparison
franchise_comparison = franchise_vs_standalone(cleaned_data)
print("Franchise vs. Standalone Comparison:")
print(franchise_comparison)

## Top Performing Movie Franchises

- **Analyze franchises:** Uses `analyze_franchise()` to group movies by franchise and calculate performance metrics.

- **Sort by total revenue:** Ranks the franchises from highest to lowest by total revenue.


In [ ]:
# Analyze franchises
franchise_stats = analyze_franchise(cleaned_data, sort_by='total_revenue_millions', ascending=False)
print("Most Successful Franchises (Sorted by Total Revenue):")
print(franchise_stats)

## Most Successful Directors

- **Analyze directors:** Groups movies by director and calculates total revenue and average performance.

- **Sort by total revenue:** Highlights top directors based on the combined earnings of their movies.


In [ ]:
# Analyze directors
director_stats = analyze_directors(cleaned_data, sort_by='total_revenue_millions', ascending=False)
print("Most Successful Directors (Sorted by Total Revenue):")
print(director_stats)

## Generating and Saving Key Movie Visualizations

1. **Revenue vs. Budget**  
   - `plot_revenue_vs_budget(cleaned_data)`  
   - Visualizes the relationship between a movie's production budget and its revenue.
   - Helps assess how effectively budgets translate into earnings.

2. **ROI by Genre**  
   - `plot_roi_by_genre(cleaned_data)`  
   - Shows the average Return on Investment (ROI) for each movie genre.
   - Useful for identifying genres that are more financially efficient.

3. **Popularity vs. Rating**  
   - `plot_popularity_vs_rating(cleaned_data)`  
   - Compares TMDB popularity scores with user ratings.
   - Reveals whether highly rated movies are also the most popular.

4. **Yearly Box Office Trends**  
   - `plot_yearly_box_office(cleaned_data)`  
   - Illustrates total revenue generated by movies per year.
   - Provides insight into box office growth or decline over time.

5. **Franchise vs. Standalone Movies**  
   - `plot_franchise_vs_standalone(cleaned_data)`  
   - Compares performance metrics (e.g., revenue, rating) between franchise and standalone films.
   - Highlights the value of sequels and shared universes.



In [ ]:
# Generate visualizations
plot_revenue_vs_budget(cleaned_data)
plt.savefig('revenue_vs_budget.png')
plt.show()

plot_roi_by_genre(cleaned_data)
plt.savefig('roi_by_genre.png')
plt.show()

plot_popularity_vs_rating(cleaned_data)
plt.savefig('popularity_vs_rating.png')
plt.show()

plot_yearly_box_office(cleaned_data)
plt.savefig('yearly_box_office.png')
plt.show()

plot_franchise_vs_standalone(cleaned_data)
plt.savefig('franchise_vs_standalone.png')
plt.show()

print("Visualizations saved as PNG files:")
print("- revenue_vs_budget.png")
print("- roi_by_genre.png")
print("- popularity_vs_rating.png")
print("- yearly_box_office.png")
print("- franchise_vs_standalone.png")